In [ ]:
%load_ext autoreload
%autoreload 2


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

from service.finetune.finetune_chest_xray import finetune


In [2]:
os.makedirs("../_results_chest_xray/finetune", exist_ok=True)
model_names = [
    "resnet50",
    "alexnet",
    "densenet121",
    "mobilenet_v3_small",
    "vgg16",
]


In [3]:
def save_figure(x, y, xlabel, ylabel, title, model_name, filename):
    fig, ax = plt.subplots()
    ax = sns.lineplot(x=x, y=y, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.figure.savefig(f"../_results_chest_xray/finetune/{model_name}/{filename}.png")
    plt.close()


In [4]:
for index, model_name in tqdm(enumerate(model_names), position=0, leave=True):
    os.makedirs(f"../_results_chest_xray/finetune/{model_name}", exist_ok=True)
    try:
        model = torch.hub.load(
            "pytorch/vision:v0.13.1", model_name, weights="IMAGENET1K_V2"
        )
    except (ValueError, KeyError):
        model = torch.hub.load(
            "pytorch/vision:v0.13.1", model_name, weights="IMAGENET1K_V1"
        )

    # Due to limited gpu memory for densenet model
    if model_name == "densenet121":
        batch_size = 32
        num_epochs = 3
    else:
        batch_size = 64
        num_epochs = 3

    # Fine-tune for 2 classes (NORMAL, PNEUMONIA)
    train_acc_history, train_loss_history, final_accuracy = finetune(
        model,
        num_classes=2,
        batch_size=batch_size,
        num_epochs=num_epochs,
        feature_extract=False,
    )

    with open(f"../_results_chest_xray/finetune/{model_name}/results.txt", "w") as f:
        for index, (loss, accuracy) in enumerate(
            zip(train_loss_history, train_acc_history)
        ):
            f.write(f"Epoch: {index + 1}, Loss: {loss}, Accuracy: {accuracy}\n")
        f.write(f"\nFinal Validation Accuracy: {final_accuracy}\n")

    save_figure(
        x=range(len(train_acc_history)),
        y=train_acc_history,
        xlabel="epoch",
        ylabel="accuracy",
        title=f"{model_name} accuracy data",
        model_name=model_name,
        filename="accuracy",
    )
    save_figure(
        x=range(len(train_loss_history)),
        y=train_loss_history,
        xlabel="epoch",
        ylabel="loss",
        title=f"{model_name} loss data",
        model_name=model_name,
        filename="loss",
    )

    model = model.cpu()
    torch.onnx.export(
        model,
        torch.ones(1, 3, 224, 224),
        f"../_results_chest_xray/finetune/{model_name}/model_ft.onnx",
    )
    del model
    torch.cuda.empty_cache()


0it [00:00, ?it/s]Using cache found in /home/shafigh/.cache/torch/hub/pytorch_vision_v0.13.1


Epoch 1/3
----------


100%|██████████| 146/146 [01:16<00:00,  1.90it/s]


Loss: 0.3698 Accuracy: 0.8619
Epoch 2/3
----------


100%|██████████| 146/146 [01:16<00:00,  1.91it/s]


...

100%|██████████| 146/146 [02:23<00:00,  1.02it/s]


Loss: 0.1022 Accuracy: 0.9621
Epoch 3/3
----------


100%|██████████| 146/146 [02:23<00:00,  1.02it/s]


Loss: 0.0634 Accuracy: 0.9762
Training complete in 7m 22s
Final Accuracy is: 0.969469
[torch.onnx] Obtain model graph for `VGG([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VGG([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


5it [17:36, 211.24s/it]
